# Audio Deepfake Detector v2
**Base**: facebook/wav2vec2-base + LoRA | **Output**: saghi776/aiscern-audio-detector
**Target**: 80%+ accuracy | **Platform**: Kaggle T4 x2 | ~4h

Datasets: ASVspoof/WaveFake/MLAAD/LibriSpeech
Wire into hf-analyze.ts MODELS.audio_finetuned after push.
Labels: {0:'real', 1:'fake'}

In [ ]:
!pip install -q transformers==4.41.0 datasets peft==0.11.0 accelerate==0.30.0 evaluate scikit-learn huggingface_hub librosa soundfile
import warnings, logging
warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)
print('packages ok')

In [ ]:
import os, torch, random, numpy as np
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN','')

BASE_MODEL   = 'facebook/wav2vec2-base'
PUSH_REPO    = 'saghi776/aiscern-audio-detector'
CKPT_DIR     = '/kaggle/working/audio-ckpt'
SR           = 16000
MAX_SAMP     = SR * 6         # 6 seconds
N_PER_CLASS  = 35000
BATCH        = 8
GRAD_ACC     = 8
EPOCHS       = 6
LR           = 1e-4
LORA_R       = 8
SEED         = 42
random.seed(SEED); np.random.seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
from datasets import load_dataset, Audio as HFAudio
import librosa, numpy as np

real_clips, fake_clips = [], []

def proc(row, col='audio'):
    a = row.get(col, {})
    if not isinstance(a, dict): return None
    arr = np.array(a.get('array', []), dtype=np.float32)
    sr  = a.get('sampling_rate', SR)
    if len(arr) < SR // 2: return None
    if sr != SR: arr = librosa.resample(arr, orig_sr=sr, target_sr=SR)
    arr = arr[:MAX_SAMP]
    if len(arr) < MAX_SAMP: arr = np.pad(arr, (0, MAX_SAMP - len(arr)))
    return arr.astype(np.float32)

# 1. LibriSpeech (real)
print('Loading LibriSpeech...')
try:
    ls = load_dataset('openslr/librispeech_asr', 'clean', split='train.100', trust_remote_code=True)
    ls = ls.cast_column('audio', HFAudio(sampling_rate=SR))
    for row in ls.select(range(min(len(ls), 20000))):
        a = proc(row)
        if a is not None: real_clips.append(a)
    print(f'  LibriSpeech real: {len(real_clips):,}')
except Exception as e: print(f'  LibriSpeech failed: {e}')

# 2. MLAAD
print('Loading MLAAD...')
try:
    ml = load_dataset('nisargsoni/MLAAD', split='train', trust_remote_code=True)
    ml = ml.cast_column('audio', HFAudio(sampling_rate=SR))
    for row in ml.select(range(min(len(ml), 25000))):
        a = proc(row)
        lbl = str(row.get('label','')).lower()
        if a is None: continue
        if 'fake' in lbl or 'spoof' in lbl or lbl == '1': fake_clips.append(a)
        else: real_clips.append(a)
    print(f'  MLAAD fake: {len(fake_clips):,}')
except Exception as e: print(f'  MLAAD failed: {e}')

# 3. WaveFake
print('Loading WaveFake...')
try:
    wf = load_dataset('m-bain/WaveFake', split='train', trust_remote_code=True)
    wf = wf.cast_column('audio', HFAudio(sampling_rate=SR))
    for row in wf.select(range(min(len(wf), 20000))):
        a = proc(row)
        if a is None: continue
        if int(row.get('label', 1)) == 1: fake_clips.append(a)
        else: real_clips.append(a)
    print(f'  WaveFake: {len(fake_clips):,} fake total')
except Exception as e: print(f'  WaveFake failed: {e}')

print(f'Total real: {len(real_clips):,} | fake: {len(fake_clips):,}')

In [ ]:
# Balance + create Dataset
real_s = random.sample(real_clips, min(len(real_clips), N_PER_CLASS))
fake_s = random.sample(fake_clips, min(len(fake_clips), N_PER_CLASS))
all_data = [(a,0) for a in real_s] + [(a,1) for a in fake_s]
random.shuffle(all_data)
split = int(len(all_data)*0.90)
train_data, val_data = all_data[:split], all_data[split:]
print(f'Train: {len(train_data):,} | Val: {len(val_data):,}')

class AudioDS(torch.utils.data.Dataset):
    def __init__(self, d): self.d = d
    def __len__(self): return len(self.d)
    def __getitem__(self, i):
        a, l = self.d[i]
        return {'input_values': torch.tensor(a, dtype=torch.float32),
                'labels': torch.tensor(l, dtype=torch.long)}

train_ds = AudioDS(train_data)
val_ds   = AudioDS(val_data)
print('Dataset ready')

In [ ]:
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2FeatureExtractor
from peft import LoraConfig, get_peft_model, TaskType

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(BASE_MODEL, token=HF_TOKEN)
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=2,
    id2label={0:'real', 1:'fake'},
    label2id={'real':0, 'fake':1},
    token=HF_TOKEN,
)
model.freeze_feature_encoder()

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS, r=LORA_R, lora_alpha=LORA_R*2,
    lora_dropout=0.05, bias='none',
    target_modules=['q_proj','k_proj','v_proj','out_proj'],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import numpy as np

def collate(batch):
    return {'input_values': torch.stack([b['input_values'] for b in batch]),
            'labels': torch.stack([b['labels'] for b in batch])}

def metrics(ep):
    logits, labels = ep
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:,1].numpy()
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds),
            'f1': f1_score(labels, preds),
            'roc_auc': roc_auc_score(labels, probs)}

args = TrainingArguments(
    output_dir=CKPT_DIR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC, learning_rate=LR,
    weight_decay=0.01, warmup_ratio=0.05, lr_scheduler_type='cosine',
    evaluation_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='f1',
    fp16=torch.cuda.is_available(), logging_steps=100,
    report_to='none', remove_unused_columns=False, seed=SEED,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=metrics, data_collator=collate,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
print('Training...')
trainer.train()

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt, seaborn as sns

res = trainer.evaluate()
acc, f1, auc = res.get('eval_accuracy',0), res.get('eval_f1',0), res.get('eval_roc_auc',0)
print(f'Accuracy: {acc*100:.2f}% | F1: {f1*100:.2f}% | AUC: {auc*100:.2f}%')
print('PASSED' if acc>=0.80 else 'Below target - add more data')

po = trainer.predict(val_ds)
preds = np.argmax(po.predictions, axis=-1)
labels = po.label_ids
cm = confusion_matrix(labels, preds)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=['Real','Fake'], yticklabels=['Real','Fake'], cmap='Oranges')
plt.title(f'Audio Acc={acc*100:.1f}%')
plt.tight_layout(); plt.show()
print(classification_report(labels, preds, target_names=['real','fake']))

merged = model.merge_and_unload()
merged.push_to_hub(PUSH_REPO, token=HF_TOKEN)
feature_extractor.push_to_hub(PUSH_REPO, token=HF_TOKEN)
print(f'Live at https://huggingface.co/{PUSH_REPO}')
print('Add to hf-analyze.ts: audio_finetuned: saghi776/aiscern-audio-detector with weight 0.50')